In [1]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    display(f"Loaded config from: {config_path}")
    display(f"Base URL: {api_base_url}")
    display(f"Access ID: {api_access_id}")
    display(f"Default Org: {api_default_org}")
except Exception as e:
    display(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    display("ThreatConnect initialized.")
except Exception as e:
    display(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    display("RequestObject successfully created.")
except Exception as e:
    display(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




'Loaded config from: C:\\Users\\jaskew\\Documents\\project_repository\\scripts\\Data Movement\\ThrearConnect-api-pull\\utils\\config.json'

'Base URL: https://hvs.threatconnect.com/api'

'Access ID: 09783848890162390382'

'Default Org: HTOC Org'

'ThreatConnect initialized.'

'RequestObject successfully created.'

In [2]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=1000)).date()
start = f"{start_date}T00:00:00Z"
type_names = ["Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
              "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])
list_of_owners = ['HTOC Org']
final_results = []

# Query indicators
for owner in list_of_owners:
    display(f"Querying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition}) AND '
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators?tql={tql_encoded}&fields=tags,observations,associatedGroups,falsePositives,threatAssess,&resultStart=0&resultLimit=10000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
    except Exception as e:
        display(f"Failed to query indicators for {owner}: {e}")

# Normalize results
normalized_data = []
for result in final_results:
    data_items = result.get('data', [])
    if not data_items:
        display("No data returned in API response:", result)
    for item in data_items:
        if isinstance(item, dict) and 'summary' in item:
            normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].str.split().str[0].str.strip()
    observed_src.drop_duplicates(subset='indicator', inplace=True)
    observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True)
    observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start)]
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)


'Querying owner: HTOC Org'

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,threatAssessRating,...,address,source,sha256,url,text,md5,lastFalsePositive,sha1,size,indicator
0,6755399457468485,2025-06-06T13:12:09Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-12-01T12:36:59Z,3.0,37.0,3.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,link.mail.beehiiv.com
1,6755399460008511,2025-07-02T12:05:38Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-01T12:07:00Z,1.0,52.0,1.25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,88.80.26.4
2,5629499536037716,2025-04-15T17:38:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-01T12:01:59Z,5.0,69.0,2.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,216.131.84.178
3,6755399459033768,2025-06-16T17:42:52Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-01T11:07:00Z,3.0,57.0,3.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45.138.16.164
4,6755399459033762,2025-06-16T17:42:44Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-01T11:07:00Z,3.0,66.0,2.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,192.42.116.217
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3813,223630,2016-12-13T19:56:35Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,URL,2023-05-18T20:50:10Z,3.0,89.0,3.00,...,NaN,NaN,NaN,NaN,http://www.zenpad.org/,NaN,NaN,NaN,NaN,http://www.zenpad.org/
3814,223570,2016-12-13T19:56:35Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,URL,2023-03-31T18:50:10Z,3.0,89.0,3.00,...,NaN,NaN,NaN,NaN,http://www.fultonarts.org/index.php/art-centers,NaN,NaN,NaN,NaN,http://www.fultonarts.org/index.php/art-centers
3815,4312413,2023-03-27T12:00:37Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,File,2023-03-28T12:00:42Z,5.0,100.0,5.00,...,NaN,API ingestion - R&F,4624F78645A6D9DF96212C243A38344B7C3859038278B6...,NaN,NaN,913283BB9EB72DFAD5005436B07BBD38,NaN,86D738AF8DCA223C6A4040F4AB0A066D898696D9,327719.0,913283BB9EB72DFAD5005436B07BBD38
3816,4307013,2023-03-12T12:00:16Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,File,2023-03-12T12:00:26Z,5.0,100.0,5.00,...,NaN,API ingestion - R&F,7B7F664680D821D8A04DA0B59DE3366E53FA6227070571...,NaN,NaN,FD1390C116CE2AEB7F41F4841A8898A7,NaN,18EF5830224B965E96669150516C151FD449D415,808599.0,FD1390C116CE2AEB7F41F4841A8898A7


In [4]:
observed_src[observed_src['indicator'] == '190.202.131.222']

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,threatAssessRating,...,address,source,sha256,url,text,md5,lastFalsePositive,sha1,size,indicator
1967,5629499542033447,2025-05-28T15:24:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-26T08:36:19Z,3.0,54.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,190.202.131.222


In [8]:

# Pull data from Z drive Excel sheet
import pandas as pd

excel_path = r"Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores\Threat_Assessment_Scores.xlsx"
sheet_name = 'Score Comparison'

try:
    print(f"Attempting to read from: {excel_path}")
    df_scores = pd.read_excel(excel_path, sheet_name=sheet_name)
    display(df_scores.head())
    print(f"Successfully loaded {len(df_scores)} rows.")
except Exception as e:
    print(f"Error reading file: {e}")

Attempting to read from: Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores\Threat_Assessment_Scores.xlsx


,Indicator,ThreatConnect Threat Score,PRISM Score,Difference
0,1-you.njalla.no,590,163,-427
1,101.71.130.99,448,423,-25
2,102.0.5.152,288,199,-89
3,102.209.18.96,283,199,-84
4,103.114.96.246,288,199,-89


Successfully loaded 1689 rows.


In [9]:
# Check ThreatConnect Threat Score and update from observed_src if it equals 0
if 'df_scores' in locals() and 'observed_src' in locals() and not observed_src.empty:
    # Ensure ThreatConnect Threat Score is numeric
    df_scores['ThreatConnect Threat Score'] = pd.to_numeric(df_scores['ThreatConnect Threat Score'], errors='coerce').fillna(0)
    
    # Find rows where ThreatConnect Threat Score equals 0
    zero_score_mask = df_scores['ThreatConnect Threat Score'] == 0
    
    print(f"Found {zero_score_mask.sum()} indicators with ThreatConnect Threat Score = 0")
    
    # Create a mapping dictionary from observed_src: indicator -> threatAssessScore
    if 'threatAssessScore' in observed_src.columns:
        # Ensure threatAssessScore is numeric
        observed_src['threatAssessScore'] = pd.to_numeric(observed_src['threatAssessScore'], errors='coerce').fillna(0)
        
        # Create mapping dictionary
        score_mapping = dict(zip(observed_src['indicator'], observed_src['threatAssessScore']))
        
        # Update scores where ThreatConnect Threat Score is 0
        updated_count = 0
        for idx in df_scores[zero_score_mask].index:
            indicator = df_scores.loc[idx, 'Indicator']
            if indicator in score_mapping:
                new_score = score_mapping[indicator]
                if new_score != 0:  # Only update if we found a non-zero score
                    df_scores.loc[idx, 'ThreatConnect Threat Score'] = new_score
                    updated_count += 1
        
        print(f"Updated {updated_count} indicators with scores from observed_src")
        
        # Display some examples of updated rows
        if updated_count > 0:
            updated_rows = df_scores[df_scores.index.isin(df_scores[zero_score_mask].index) & 
                                     (df_scores['ThreatConnect Threat Score'] != 0)]
            if len(updated_rows) > 0:
                display_cols = ['Indicator', 'ThreatConnect Threat Score']
                display_cols = [col for col in display_cols if col in updated_rows.columns]
                print("\nSample of updated indicators:")
                display(updated_rows[display_cols].head(10))
    else:
        print("Warning: 'threatAssessScore' column not found in observed_src")
        print(f"Available columns in observed_src: {list(observed_src.columns)}")
else:
    if 'df_scores' not in locals():
        print("Error: df_scores not found. Please run the previous cell first.")
    if 'observed_src' not in locals():
        print("Error: observed_src not found. Please run Cell 1 first.")
    if 'observed_src' in locals() and observed_src.empty:
        print("Warning: observed_src is empty. No data to update from.")


Found 610 indicators with ThreatConnect Threat Score = 0
Updated 586 indicators with scores from observed_src

Sample of updated indicators:


,Indicator,ThreatConnect Threat Score
1079,190.121.154.147,283
1080,190.131.227.30,288
1081,194.230.160.237,585
1082,197.219.228.62,283
1083,bimbinlombardia.com,368
1084,freeweddingpsd.com,368
1085,hcmiu.edu.vn,358
1086,metalhoz.com,368
1087,r3.o.lencr.org,288
1088,svgarchive.com,368


In [10]:
# Save updated scores back to Excel file
import openpyxl
from openpyxl import load_workbook

if 'df_scores' in locals():
    excel_path = r"Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores\Threat_Assessment_Scores.xlsx"
    
    try:
        # Read all existing sheets first
        excel_file = pd.ExcelFile(excel_path, engine='openpyxl')
        all_sheets = {}
        
        # Store all existing sheets
        for sheet_name in excel_file.sheet_names:
            if sheet_name != 'Score Comparison':
                all_sheets[sheet_name] = pd.read_excel(excel_path, sheet_name=sheet_name, engine='openpyxl')
        
        # Close the ExcelFile object
        excel_file.close()
        
        # Now write all sheets back, including the updated Score Comparison sheet
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            # Write the updated Score Comparison sheet
            df_scores.to_excel(writer, index=False, sheet_name='Score Comparison')
            
            # Write all other existing sheets back
            for sheet_name, sheet_df in all_sheets.items():
                sheet_df.to_excel(writer, index=False, sheet_name=sheet_name)
        
        print(f"Successfully updated 'Score Comparison' sheet in {excel_path}")
        print(f"Updated {len(df_scores)} rows with corrected ThreatConnect Threat Scores")
        
        # Verify the update
        verify_df = pd.read_excel(excel_path, sheet_name='Score Comparison', engine='openpyxl')
        zero_count = (pd.to_numeric(verify_df['ThreatConnect Threat Score'], errors='coerce').fillna(0) == 0).sum()
        print(f"Remaining indicators with ThreatConnect Threat Score = 0: {zero_count}")
        print(f"Preserved {len(all_sheets)} other sheet(s): {', '.join(all_sheets.keys())}")
        
    except Exception as e:
        print(f"Error saving to Excel file: {e}")
        import traceback
        traceback.print_exc()
else:
    print("Error: df_scores not found. Please run the previous cells first.")


Successfully updated 'Score Comparison' sheet in Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores\Threat_Assessment_Scores.xlsx
Updated 1689 rows with corrected ThreatConnect Threat Scores
Remaining indicators with ThreatConnect Threat Score = 0: 24
Preserved 1 other sheet(s): PRISM Scores
